# 04 — Improved Model (v2) — 5 soumissions

**Score précédent: AUC=0.8008, F1=0.6985, Zindi=0.7394**

Ce notebook génère **5 fichiers de soumission** :

| Fichier | Modèle |
|---|---|
| `v2_lgb.csv` | LightGBM (seuil OOF optimal) |
| `v2_xgb.csv` | XGBoost (seuil OOF optimal) |
| `v2_rf.csv` | Random Forest (seuil OOF optimal) |
| `v2_cb.csv` | CatBoost — optionnel (`uv add catboost`) |
| `v2_blend.csv` | Ensemble pondéré LGB+XGB+RF (+CB si dispo) |

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, roc_auc_score

from src.features import prepare_Xy
from src.train import cross_validate, blend, save_models
from src.evaluate import find_best_threshold, print_scores
from src.predict import make_submission, load_threshold, predict_proba, generate_all_submissions

DATA = pathlib.Path('..') / 'data' / 'raw'
MODEL_DIR = pathlib.Path('..') / 'models'
MODEL_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## 1. Feature engineering (177 features)

In [ ]:
X_train, y_train, X_test = prepare_Xy(DATA / 'Train.csv', DATA / 'Test.csv')
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
new_feats = [c for c in X_train.columns if any(x in c for x in ['trend', '_cv', 'water_score', 'pond_signal', 'vh_ndwi'])]
print(f'Nouvelles features v2 ({len(new_feats)}): {new_feats}')

## 2. Modèle 1 — LightGBM

In [ ]:
print('=== LightGBM ===')
lgb_oof, lgb_models, lgb_thr = cross_validate(X_train, y_train, model_type='lgb')
save_models(lgb_models, 'lgb')

## 3. Modèle 2 — XGBoost

In [ ]:
print('=== XGBoost ===')
xgb_oof, xgb_models, xgb_thr = cross_validate(X_train, y_train, model_type='xgb')
save_models(xgb_models, 'xgb')

## 4. Modèle 3 — Random Forest

In [ ]:
print('=== Random Forest ===')
rf_oof, rf_models, rf_thr = cross_validate(X_train, y_train, model_type='rf')
save_models(rf_models, 'rf')

## 5. Modèle 4 — CatBoost (optionnel)

In [ ]:
try:
    import catboost
    print('=== CatBoost ===')
    cb_oof, cb_models, cb_thr = cross_validate(X_train, y_train, model_type='cb')
    save_models(cb_models, 'cb')
    HAS_CB = True
except ImportError:
    print('CatBoost non installé — run: uv add catboost')
    HAS_CB = False
    cb_oof, cb_thr = None, None

## 6. Blend + seuil optimal OOF

In [ ]:
oof_list   = [lgb_oof, xgb_oof, rf_oof]
w_list     = [3, 2, 2]
label_list = ['LGB', 'XGB', 'RF']
if HAS_CB:
    oof_list.append(cb_oof); w_list.append(2); label_list.append('CB')

blended_oof = blend(oof_list, weights=w_list)
blend_thr, blend_f1 = find_best_threshold(y_train.values, blended_oof)

print('=' * 52)
print(f'Blend ({" + ".join(label_list)})  seuil=0.50')
print_scores(y_train.values, blended_oof, threshold=0.50)
print(f'\nBlend ({" + ".join(label_list)})  seuil optimal={blend_thr:.3f}')
print_scores(y_train.values, blended_oof, threshold=blend_thr)
print('=' * 52)

# Sauvegarder les seuils
thresholds_dict = {'blend': blend_thr, 'lgb': lgb_thr, 'xgb': xgb_thr, 'rf': rf_thr}
if HAS_CB:
    thresholds_dict['cb'] = cb_thr
with open(MODEL_DIR / 'thresholds.json', 'w') as f:
    json.dump(thresholds_dict, f, indent=2)
print('\nSeuils sauvegardés:', thresholds_dict)

## 7. Sweep threshold — visualisation

In [ ]:
thresholds = np.linspace(0.05, 0.95, 200)
auc_val = roc_auc_score(y_train.values, blended_oof)
f1_vals = [f1_score(y_train.values, (blended_oof >= t).astype(int), zero_division=0) for t in thresholds]
zindi_vals = [0.6 * f + 0.4 * auc_val for f in f1_vals]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, f1_vals,    label='F1',    color='steelblue')
ax.plot(thresholds, zindi_vals, label='Zindi', color='coral')
ax.axvline(blend_thr, color='green', ls='--', label=f'Optimal={blend_thr:.3f}')
ax.axvline(0.5,       color='gray',  ls=':',  label='Défaut=0.5')
ax.set_xlabel('Seuil'); ax.legend()
ax.set_title('Sweep seuil — prédictions OOF blend')
plt.tight_layout(); plt.show()
print(f'Gain F1 par optimisation seuil: +{max(f1_vals) - f1_vals[99]:.4f}')

## 8. Comparaison OOF entre modèles

In [ ]:
models_oof = {'LGB': (lgb_oof, lgb_thr), 'XGB': (xgb_oof, xgb_thr), 'RF': (rf_oof, rf_thr)}
if HAS_CB:
    models_oof['CB'] = (cb_oof, cb_thr)
models_oof['Blend'] = (blended_oof, blend_thr)

print(f'{"Modèle":<10} {"F1":>8} {"AUC":>8} {"Zindi":>8} {"Seuil":>8}')
print('-' * 50)
for name, (oof, thr) in models_oof.items():
    f1  = f1_score(y_train.values, (oof >= thr).astype(int), zero_division=0)
    auc = roc_auc_score(y_train.values, oof)
    z   = 0.6 * f1 + 0.4 * auc
    print(f'{name:<10} {f1:>8.4f} {auc:>8.4f} {z:>8.4f} {thr:>8.3f}')

## 9. Importance features LGB (top 30)

In [ ]:
importances = np.stack([m.booster_.feature_importance(importance_type='gain') for m in lgb_models])
feat_imp = pd.Series(importances.mean(axis=0), index=lgb_models[0].feature_name_).sort_values(ascending=False)
top30 = feat_imp.head(30).sort_values()
colors = ['#DD8452' if any(x in c for x in ['trend', '_cv', 'water_score', 'pond_signal', 'vh_ndwi'])
          else '#4C72B0' for c in top30.index]
fig, ax = plt.subplots(figsize=(9, 8))
top30.plot.barh(ax=ax, color=colors)
ax.set_title('Top-30 LGB — orange = nouvelles features v2')
plt.tight_layout(); plt.show()

## 10. Génération des 5 fichiers de soumission

In [ ]:
print('Génération des 5 soumissions...\n')
probas = generate_all_submissions(X_test, tag='v2')

print('\n=== Récap fichiers générés ===')
for f in sorted((pathlib.Path('..') / 'data' / 'submissions').glob('v2_*.csv')):
    df = pd.read_csv(f)
    n_pos = df['TargetF1'].sum()
    print(f'  {f.name:45s}  positifs: {n_pos:3d} / {len(df)}')